<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/03_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 3: 🤗 Datasets

Data is half of machine learning. The `datasets` library loads a dataset from the Hub with one line, lets you look at it and change it, and shares your own datasets. At the end we take a quick look at **tokenizers**, which turn the texts into tokens for a model.

**Contents:** 0. Setup · 1. Loading a dataset · 2. Looking inside · 3. `filter`, `map` and friends · 4. Streaming huge datasets · 5. Your own datasets on the Hub · 6. Tokenizers · 7. Summary

No GPU needed. For section 5 you need the `HF_TOKEN` secret from notebook 1.

---
## 0. Setup

In [ ]:
import time

import matplotlib.pyplot as plt
import datasets

print("datasets", datasets.__version__)

In [ ]:
# @title Helper function (just run this cell)
import html
from IPython.display import HTML, display

COLORS = ["#fde68a", "#bfdbfe", "#bbf7d0", "#fbcfe8", "#ddd6fe", "#fed7aa"]

def show_tokens(pieces):
    # Show text pieces in alternating colors, like the Tiktokenizer app (as in lesson 04).
    spans = []
    for i, piece in enumerate(pieces):
        piece = html.escape(piece).replace("\n", "↵\n")
        spans.append(f'<span style="background:{COLORS[i % len(COLORS)]}; color:black; border-radius:3px">{piece}</span>')
    display(HTML(f'<pre style="font-size:15px; line-height:1.8; white-space:pre-wrap; margin:0">{"".join(spans)}</pre>'
                 f'<div style="font-size:12px; margin-bottom:8px">{len(pieces)} tokens</div>'))

---
## 1. Loading a dataset

The Hub hosts hundreds of thousands of datasets. Our example: 30,000 Czech user reviews from [ČSFD](https://www.csfd.cz), the Czech movie database. Open its page first: [huggingface.co/datasets/fewshot-goes-multilingual/cs_csfd-movie-reviews](https://huggingface.co/datasets/fewshot-goes-multilingual/cs_csfd-movie-reviews). The **dataset viewer** there lets you scroll through the data before you download anything.

`load_dataset` downloads it (the first time) and loads it:

In [ ]:
from datasets import load_dataset

reviews = load_dataset("fewshot-goes-multilingual/cs_csfd-movie-reviews")
reviews

A `DatasetDict`: three **splits** (train, validation, test), and in each a `Dataset`, a table with named columns (*features*) and rows.

---
## 2. Looking inside

One row is a Python dictionary:

In [ ]:
train = reviews["train"]
train[0]

A slice gives a dictionary of **lists**, one list per column. So `train[:3]["comment"]` are the first three reviews, and `train["rating_int"]` is the whole column:

In [ ]:
for comment in train[:3]["comment"]:
    print(comment.strip()[:200])
print(train["rating_int"][:20])

Every column has a **type**, so the data is stored and loaded the same way on every computer:

In [ ]:
train.features

**Look inside: where is the data?** Not in memory! `datasets` stores the table in a file in the **Apache Arrow** format and reads only the rows you ask for. That is why it can open datasets much bigger than your RAM.

In [ ]:
print(train.cache_files)
!ls -lh ~/.cache/huggingface/datasets

For exploring, `pandas` is often handier. `to_pandas()` loads the whole table into a `DataFrame`:

In [ ]:
df = train.to_pandas()
print(df["item_title"].value_counts().head(10))

df["rating_int"].value_counts().sort_index().plot.bar(title="ratings (0-5 stars)")
plt.show()

The ratings are **balanced**: the authors of the dataset chose about the same number of reviews for every number of stars. On the real ČSFD there are many more good ratings than bad ones.

---
## 3. `filter`, `map` and friends

`filter` keeps the rows for which a function returns `True`:

In [ ]:
avatar = train.filter(lambda row: row["item_title"] == "Avatar")
print(len(avatar), "reviews of Avatar")
for row in avatar.select(range(3)):           # select: rows by their position
    print(row["rating_str"], row["comment"].strip()[:150])

`map` runs a function on every row. The function returns a dictionary, and its keys become **new columns**:

In [ ]:
def count_words(row):
    return {"n_words": len(row["comment"].split())}

train = train.map(count_words)
print(train)
print("average review length:", sum(train["n_words"]) / len(train), "words")

A few more useful methods:

In [ ]:
sample = train.shuffle(seed=42).select(range(1000))          # 1,000 random rows
parts = sample.train_test_split(test_size=0.2, seed=42)      # a new random split
print(parts)
print(sample.sort("n_words")[-1]["n_words"], "words in the longest review of the sample")

**Try it:**
- How many reviews does your favourite movie have, and what is its average rating?
- Do longer reviews give better or worse ratings? (Hint: `df.groupby("rating_int")["comment"].apply(lambda s: s.str.len().mean())`.)
- Which director has the best average rating? (`item_directors` is a list, `df.explode("item_directors")` makes one row per director.)

---
## 4. Streaming huge datasets

The Czech Wikipedia has 534,044 articles, about 1 GB to download (5.6 GB as an Arrow table). With `streaming=True` nothing is downloaded up front: the rows arrive one by one, while you read them.

In [ ]:
wiki = load_dataset("wikimedia/wikipedia", "20231101.cs", split="train", streaming=True)

for article in wiki.take(3):           # only the first 3 articles
    print(f"{article['title']}: {len(article['text']):,} characters")
    print("   ", article["text"][:200].replace("\n", " "))

This is how big models are trained: FineWeb, a typical LLM training set, has 15 trillion tokens, far more than fits on one disk.

---
## 5. Your own datasets on the Hub

In notebook 1 we uploaded the texts generated by GPT-2. Loading them back is the same one line as for any other dataset:

In [ ]:
from huggingface_hub import whoami

user = whoami()["name"]          # uses the HF_TOKEN secret from notebook 1
mine = load_dataset(f"{user}/gpt2-generated")
print(mine)
print(mine["train"][0])

And any dataset we make can go up too, e.g. the Avatar reviews from section 3. `load_dataset` can also read local files, e.g. `load_dataset("csv", data_files="my_table.csv")`, and then you upload them the same way.

In [ ]:
avatar.push_to_hub(f"{user}/csfd-avatar")
print(f"https://huggingface.co/datasets/{user}/csfd-avatar")

Open the link: the dataset viewer works for your dataset too. On the Hub, datasets are stored as **Parquet** files, a compressed table format (look at *Files and versions*).

---
## 6. Tokenizers

Before a model can read a text, a **tokenizer** cuts it into tokens. Last week we rebuilt GPT-2's BPE tokenizer in Python. The tokenizers you get with `AutoTokenizer` run on the `tokenizers` library, which does the same thing in Rust, a fast compiled language. (It can also train a new tokenizer on your own text in a few seconds, see the [documentation](https://huggingface.co/docs/tokenizers).)

Let's compare two tokenizers on a Czech review: GPT-2's, learned on English, and the one of **mmBERT**, a model that knows 1,800 languages (we fine-tune it in notebook 4).

In [ ]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
mmbert_tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/mmBERT-small")
print(type(gpt2_tokenizer.backend_tokenizer))       # the Rust tokenizer from the `tokenizers` library

def pieces(tokenizer, text):
    # The text cut into tokens, each token decoded back to readable text.
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return [tokenizer.decode([i]) for i in ids]

review = "Tenhle film byl naprostá nuda, ale herci hráli skvěle."
print("GPT-2:"); show_tokens(pieces(gpt2_tokenizer, review))
print("mmBERT:"); show_tokens(pieces(mmbert_tokenizer, review))

GPT-2 cuts Czech words into small pieces, often into half letters (shown as `�`: letters like `ě` take 2 bytes). mmBERT's tokenizer has 256,000 tokens for 1,800 languages, among them many pieces of Czech words (` skvě`, ` hrá`), so it needs fewer tokens and never splits a letter.

**Tokenizing a whole dataset.** Before training, we turn every text of the dataset into token ids. `map` with `batched=True` hands the function 1,000 rows at once, and the Rust tokenizer processes them in parallel:

In [ ]:
def count_tokens(batch):
    return {
        "gpt2_tokens": [len(ids) for ids in gpt2_tokenizer(batch["comment"])["input_ids"]],
        "mmbert_tokens": [len(ids) for ids in mmbert_tokenizer(batch["comment"])["input_ids"]],
    }

t0 = time.time()
train = train.map(count_tokens, batched=True)
print(f"tokenized {len(train):,} reviews twice in {time.time() - t0:.1f} s")

n_words = sum(train["n_words"])
print(f"GPT-2 tokens per word:  {sum(train['gpt2_tokens']) / n_words:.2f}")
print(f"mmBERT tokens per word: {sum(train['mmbert_tokens']) / n_words:.2f}")

*(You can ignore the warning that a sequence is longer than the maximum length: we only count the tokens, we do not give them to the model.)*

For Czech, GPT-2 needs about 1.6 times as many tokens as mmBERT: the "Czech token tax" from last week. In notebook 4 we tokenize the reviews exactly this way, and then train mmBERT to tell positive reviews from negative ones.

---
## 7. Summary

| step | code |
|---|---|
| loading a dataset | `load_dataset("owner/name")`, with `streaming=True` for huge ones |
| looking at it | `ds[0]`, `ds[:5]["column"]`, `ds.features`, `ds.to_pandas()` |
| changing it | `filter`, `map` (with `batched=True` for speed), `select`, `shuffle`, `sort`, `train_test_split` |
| your own data | `load_dataset("csv" / "json" / "text", data_files=...)`, `Dataset.from_pandas(df)`, `push_to_hub` |
| tokenizing | `AutoTokenizer.from_pretrained(...)`, `ds.map(tokenize, batched=True)` |

**New words:** dataset split, features, Apache Arrow, Parquet, streaming, dataset viewer, batched map.

### Exercises
1. Which genre (`item_genres`) has the best average rating? And which has the longest reviews?
2. Upload a dataset of your own, e.g. a CSV file from an earlier project, and look at it in the dataset viewer.
3. Find a Czech dataset on the Hub ([huggingface.co/datasets?language=language:cs](https://huggingface.co/datasets?language=language:cs)) and explore it with this notebook.

### Further reading
- [Datasets documentation](https://huggingface.co/docs/datasets), [Tokenizers documentation](https://huggingface.co/docs/tokenizers)
- Hugging Face LLM course: [chapter 5, the Datasets library](https://huggingface.co/learn/llm-course/chapter5/1) and [chapter 6, the Tokenizers library](https://huggingface.co/learn/llm-course/chapter6/1)
- [FineWeb](https://huggingface.co/spaces/HuggingFaceFW/blogpost-fineweb-v1): how a 15-trillion-token training set is made